# 02 — Kat-içi ön işleme, manuel ve otomatik arama, nested CV

Bu notebook yalnızca **development** kümesini kullanır. Bağımsız test kümesi burada hiç okunmaz.

Akış:
1. `configs/` klasöründe önceden kaydedilmiş M1–M6 ve R1–R6 adayları okunur.
2. Dış CV: `StratifiedKFold(n_splits=3, shuffle=True, random_state=31078)`.
3. Her dış eğitim kümesinde iç CV: `StratifiedKFold(n_splits=3, shuffle=True, random_state=31079)`. Manuel altı aday ve otomatik altı aday **aynı iç katlarda iki ayrı arama** olarak değerlendirilir (3 × 3 × 12 = 108 eğitim).
4. Her iç eğitimde: iç eğitim → %85 fit + %15 stratified erken durdurma izleme kümesi. Ön işleme yalnızca fit kısmında öğrenilir. Erken durdurma ve ReduceLROnPlateau izleme kümesinin BCE kaybını izler. İç doğrulama katında yalnızca AP hesaplanır.
5. Her aramanın kazananı dış eğitim kümesinde yeniden eğitilir ve dış değerlendirme kümesinde bir kez ölçülür (3 × 2 = 6 eğitim). Referans modeller aynı dış katlarda eğitilir ve ölçülür.
6. Tüm development kümesinde iç CV tekrarlanır (3 × 12 = 36 eğitim) ve nihai konfigürasyon önceden tanımlı kuralla seçilir.

In [1]:
import sys, warnings, time, json, platform
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, torch, sklearn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from sklearn.exceptions import ConvergenceWarning
from src import project as P, nested as N
P.set_global_seeds()
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 40)
print("Python", platform.python_version(), "| torch", torch.__version__, "| sklearn", sklearn.__version__,
      "| CPU:", platform.processor() or platform.machine(), "| CUDA:", torch.cuda.is_available())
print("Seedler:", P.SEEDS)

Python 3.13.18 | torch 2.15.0+cu132 | sklearn 1.9.1 | CPU: x86_64 | CUDA: False
Seedler: {'python': 31074, 'numpy': 31075, 'torch': 31076, 'train_test_split': 31077, 'outer_cv': 31078, 'inner_cv': 31079, 'early_stop_split': 31080, 'random_search': 31081, 'bootstrap': 31082, 'torch_stability_2': 31176, 'torch_stability_3': 31276}


## 1. Development verisi ve önceden kaydedilmiş konfigürasyonlar

In [2]:
X, y = P.load_development()
print("Development:", X.shape, "| pozitif oranı: %.4f" % y.mean())
MANUAL = P.load_json(P.CONFIGS / "manual_configs.json")
RANDOM = P.load_json(P.CONFIGS / "random_configs.json")
SUPP = P.load_json(P.CONFIGS / "supplementary_configs.json")
SETTINGS = P.load_json(P.CONFIGS / "training_settings.json")
cols = ["optimizer", "hidden_layers", "activation", "lr", "weight_decay", "batch_size", "scaler", "dropout", "clipping", "scheduler"]
print("Manuel adaylar"); display(pd.DataFrame(MANUAL).T[cols + ["purpose"]])
print("Otomatik adaylar (ParameterSampler)"); display(pd.DataFrame(RANDOM).T[cols])

Development: (3295, 20) | pozitif oranı: 0.1096
Manuel adaylar


,optimizer,hidden_layers,activation,lr,weight_decay,batch_size,scaler,dropout,clipping,scheduler,purpose
M1,adamw,[32],relu,0.001,0.0001,64,standard,0.0,False,True,Temel konfigürasyon
M2,adam,[32],relu,0.001,0.0001,64,standard,0.0,False,True,Optimizer etkisi (AdamW -> Adam)
M3,sgd,[32],relu,0.001,0.0001,64,standard,0.0,False,True,Optimizer etkisi (AdamW -> SGD+Momentum 0.9)
M4,adamw,[32],relu,0.001,0.0001,64,minmax,0.0,False,True,Ölçekleme etkisi (Standard -> MinMax)
M5,adamw,"[64, 32]",relu,0.001,0.0001,64,standard,0.0,False,True,"Mimari etkisi ((32,) -> (64,32))"
M6,adamw,[32],relu,0.001,0.0001,64,standard,0.0,False,False,Learning-rate-drop etkisi (scheduler kapalı)


Otomatik adaylar (ParameterSampler)


,optimizer,hidden_layers,activation,lr,weight_decay,batch_size,scaler,dropout,clipping,scheduler
R1,sgd,"[64, 32]",tanh,0.071227,0.0001,64,robust,0.2,False,True
R2,sgd,[32],relu,0.039314,0.00001,32,minmax,0.0,True,True
R3,adam,[16],relu,0.000123,0.0,32,minmax,0.0,False,True
R4,adam,[64],relu,0.000862,0.00001,128,standard,0.2,False,True
R5,adamw,"[128, 64]",relu,0.00011,0.00001,64,robust,0.0,True,True
R6,adamw,[16],tanh,0.004435,0.0,32,standard,0.2,True,True


## 2. Öğrenilebilir parametre sayısı

Ön işleme sonrası girdi boyutu, fit alt kümesinde görülen kategori sayısına bağlıdır. Bu nedenle her eğitimde gerçek girdi boyutu ve parametre sayısı ayrıca kaydedilir. Tam bağlantılı bir katman için parametre sayısı `girdi × çıktı + çıktı` (ağırlıklar + bias) formülüyle hesaplanır.

In [3]:
Z_demo = P.build_preprocessor().fit(X).transform(X.iloc[:5])
n_in = Z_demo.shape[1]
print("Tüm development üzerinde öğrenilen sözlükle girdi boyutu:", n_in)
for code, cfg in {**MANUAL, **RANDOM}.items():
    m = P.MLP(n_in, tuple(cfg["hidden_layers"]), cfg["activation"], cfg["dropout"])
    assert P.count_parameters(m) == P.n_params_formula(n_in, cfg["hidden_layers"])
    print(f"{code}: gizli katmanlar {tuple(cfg['hidden_layers'])} -> {P.count_parameters(m)} parametre")

Tüm development üzerinde öğrenilen sözlükle girdi boyutu: 62
M1: gizli katmanlar (32,) -> 2049 parametre
M2: gizli katmanlar (32,) -> 2049 parametre
M3: gizli katmanlar (32,) -> 2049 parametre
M4: gizli katmanlar (32,) -> 2049 parametre
M5: gizli katmanlar (64, 32) -> 6145 parametre
M6: gizli katmanlar (32,) -> 2049 parametre
R1: gizli katmanlar (64, 32) -> 6145 parametre
R2: gizli katmanlar (32,) -> 2049 parametre
R3: gizli katmanlar (16,) -> 1025 parametre
R4: gizli katmanlar (64,) -> 4097 parametre
R5: gizli katmanlar (128, 64) -> 16385 parametre
R6: gizli katmanlar (16,) -> 1025 parametre


## 3. Nested CV akış şeması

In [4]:
def box(ax, x, y, w, h, text, fc="#eef2f7", bold=False):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.01,rounding_size=0.015", fc=fc, ec="#4a5568", lw=1.2))
    ax.text(x + w/2, y + h/2, text, ha="center", va="center", fontsize=9, fontweight="bold" if bold else "normal", wrap=True)
def arrow(ax, x1, y1, x2, y2):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="-|>", color="#4a5568", lw=1.2))

fig, ax = plt.subplots(figsize=(11, 8)); ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
box(ax, .25, .92, .5, .06, "bank-additional.csv (4.119 satır) — duration çıkarılır\nStratified split (seed 31077)", bold=True)
box(ax, .05, .80, .42, .07, "Development %80 (n = 3.295)\nEDA, nested CV, nihai seçim ve eğitim")
box(ax, .58, .80, .37, .07, "Bağımsız test %20 (n = 824)\nKilitli tahminlerden sonra bir kez ölçülür", fc="#f7f1ee")
arrow(ax, .45, .92, .26, .87); arrow(ax, .55, .92, .76, .87)
ax.add_patch(FancyBboxPatch((.03, .25), .94, .52, boxstyle="round,pad=0.01", fc="none", ec="#a0aec0", lw=1, ls="--"))
ax.text(.5, .745, "Dış CV: 3 kat (seed 31078) — her katta outer train / outer evaluation", ha="center", fontsize=10, fontweight="bold")
arrow(ax, .26, .80, .26, .72)
box(ax, .06, .60, .88, .08, "Outer train içinde iç CV: 3 kat (seed 31079) — iki arama aynı iç katlarda\nHer iç eğitim: %85 fit (ön işleme fit + ağırlık eğitimi) | %15 izleme (BCE: erken durdurma + scheduler) | iç doğrulama: AP")
box(ax, .08, .46, .38, .09, "Manuel arama: M1–M6\n6 aday × 3 iç kat = 18 eğitim\nEn yüksek ortalama AP → manuel kazanan")
box(ax, .54, .46, .38, .09, "Otomatik arama: R1–R6\n6 aday × 3 iç kat = 18 eğitim\nEn yüksek ortalama AP → otomatik kazanan", fc="#eef7f3")
arrow(ax, .27, .60, .27, .55); arrow(ax, .73, .60, .73, .55)
box(ax, .08, .35, .38, .07, "Manuel kazanan outer train üzerinde\nyeniden eğitilir → outer evaluation'da ölçülür")
box(ax, .54, .35, .38, .07, "Otomatik kazanan outer train üzerinde\nyeniden eğitilir → outer evaluation'da ölçülür", fc="#eef7f3")
arrow(ax, .27, .46, .27, .42); arrow(ax, .73, .46, .73, .42)
box(ax, .06, .27, .88, .045, "Referans modeller (Dummy, Logistic Regression) aynı dış katlarda: outer train'de eğitim, outer evaluation'da ölçüm")
box(ax, .05, .12, .9, .09, "Tüm development'ta iç CV tekrarı (3 kat × 12 aday = 36 eğitim) → nihai konfigürasyon\n3 ağırlık seed'i ile nihai eğitim → etiketsiz test tahminleri ayrı commit → test metrikleri", fc="#f2f2f2")
box(ax, .05, .02, .9, .07, "Dış CV sonuçları: her yöntem için 3 kat üzerinden Mean ± SD (n = 3)", fc="#f2f2f2")
arrow(ax, .5, .25, .5, .21)
fig.tight_layout(); fig.savefig(P.FIGURES / "fig_nested_cv_flow.png", dpi=200); plt.close(fig)

## 4. Nested CV (3 dış kat × 3 iç kat × 12 aday)

In [5]:
histories, inner_runs, outer_rows, manual_tables, random_tables = [], [], [], [], []
lr_warnings = []
t_start = time.perf_counter()
for k, (otr, oev) in enumerate(N.outer_cv().split(X, y), start=1):
    Xo, yo, Xe, ye = X.iloc[otr], y.iloc[otr], X.iloc[oev], y.iloc[oev]
    splits = list(N.inner_cv().split(Xo, yo))   # iki arama için aynı iç katlar
    ctx = {"stage": "outer", "outer_fold": k}
    runs_m = N.run_inner_search(Xo, yo, MANUAL, splits, {**ctx, "search": "manual"}, histories)
    runs_r = N.run_inner_search(Xo, yo, RANDOM, splits, {**ctx, "search": "random"}, histories)
    inner_runs += [runs_m, runs_r]
    sm, sr = N.summarize_search(runs_m, MANUAL), N.summarize_search(runs_r, RANDOM)
    win_m, win_r = N.select_candidate(sm), N.select_candidate(sr)
    sm.insert(0, "outer_fold", k); sr.insert(0, "outer_fold", k)
    sm["selected"] = sm.index == win_m; sr["selected"] = sr.index == win_r
    manual_tables.append(sm); random_tables.append(sr)
    for method, code, cfgs in [("Manual search", win_m, MANUAL), ("Random search", win_r, RANDOM)]:
        cfg = {kk: v for kk, v in cfgs[code].items() if kk != "purpose"}
        m = P.train_mlp(Xo, yo, cfg)
        t0 = time.perf_counter(); prob = m.predict_proba(Xe); t_pred = time.perf_counter() - t0
        h = m.history.copy(); h["stage"] = "outer_refit"; h["outer_fold"] = k; h["candidate"] = code; histories.append(h)
        outer_rows.append({"outer_fold": k, "method": method, "selected_candidate": code,
                           "n_outer_train": len(otr), "n_outer_eval": len(oev),
                           **P.compute_metrics(ye, prob), "fit_time_s": m.info["train_time_s"],
                           "predict_time_s": t_pred, "best_epoch": m.info["best_epoch"],
                           "stopped_epoch": m.info["stopped_epoch"], "warnings": m.info["warnings"]})
    for name in ["Dummy", "LogReg"]:
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always", ConvergenceWarning)
            model, t_fit = N.fit_reference(name, Xo, yo)
        msgs = [str(x.message) for x in w if issubclass(x.category, ConvergenceWarning)]
        t0 = time.perf_counter(); prob = model.predict_proba(Xe)[:, 1]; t_pred = time.perf_counter() - t0
        outer_rows.append({"outer_fold": k, "method": name, "selected_candidate": "-",
                           "n_outer_train": len(otr), "n_outer_eval": len(oev),
                           **P.compute_metrics(ye, prob), "fit_time_s": t_fit, "predict_time_s": t_pred,
                           "best_epoch": np.nan, "stopped_epoch": np.nan, "warnings": " | ".join(msgs)})
    print(f"Dış kat {k}: manuel kazanan {win_m} (iç AP {sm.loc[win_m, 'mean_AP']:.4f}), "
          f"otomatik kazanan {win_r} (iç AP {sr.loc[win_r, 'mean_AP']:.4f}) — geçen süre {time.perf_counter() - t_start:.0f} s")
print("Toplam iç CV eğitimi:", sum(len(r) for r in inner_runs))

Dış kat 1: manuel kazanan M5 (iç AP 0.4277), otomatik kazanan R1 (iç AP 0.4361) — geçen süre 44 s


Dış kat 2: manuel kazanan M4 (iç AP 0.4678), otomatik kazanan R6 (iç AP 0.4835) — geçen süre 77 s


Dış kat 3: manuel kazanan M1 (iç AP 0.3705), otomatik kazanan R4 (iç AP 0.3755) — geçen süre 118 s
Toplam iç CV eğitimi: 108


In [6]:
manual_outer = pd.concat(manual_tables); manual_outer.index.name = "candidate"
random_outer = pd.concat(random_tables); random_outer.index.name = "candidate"
outer_res = pd.DataFrame(outer_rows)
outer_res.to_csv(P.RESULTS / "outer_cv_results.csv", index=False)
inner_outer_runs = pd.concat(inner_runs, ignore_index=True)
print("Dış kat sonuçları (kat bazlı, eşik 0.50)")
show = ["outer_fold", "method", "selected_candidate", "AP", "ROC_AUC", "F1", "Sensitivity", "Specificity",
        "Balanced_Accuracy", "Accuracy", "Precision", "TP", "FP", "TN", "FN", "fit_time_s"]
outer_res[show].round(4)

Dış kat sonuçları (kat bazlı, eşik 0.50)


,outer_fold,method,selected_candidate,AP,ROC_AUC,F1,Sensitivity,Specificity,Balanced_Accuracy,Accuracy,Precision,TP,FP,TN,FN,fit_time_s
0,1,Manual search,M5,0.4620,0.7718,0.3333,0.2314,0.9806,0.6060,0.8981,0.5957,28,19,959,93,0.8912
1,1,Random search,R1,0.4576,0.7707,0.3580,0.2397,0.9877,0.6137,0.9054,0.7073,29,12,966,92,0.6591
2,1,Dummy,-,0.1101,0.5000,0.0000,0.0000,1.0000,0.5000,0.8899,0.0000,0,0,978,121,0.0005
3,1,LogReg,-,0.4647,0.7703,0.3636,0.2479,0.9857,0.6168,0.9045,0.6818,30,14,964,91,0.0361
4,2,Manual search,M4,0.3736,0.7409,0.1678,0.1000,0.9888,0.5444,0.8916,0.5217,12,11,967,108,0.5780
5,2,Random search,R6,0.3869,0.7481,0.2927,0.2000,0.9796,0.5898,0.8944,0.5455,24,20,958,96,0.9224
6,2,Dummy,-,0.1093,0.5000,0.0000,0.0000,1.0000,0.5000,0.8907,0.0000,0,0,978,120,0.0005
7,2,LogReg,-,0.3832,0.7347,0.3023,0.2167,0.9734,0.5950,0.8907,0.5000,26,26,952,94,0.0295
8,3,Manual search,M1,0.4843,0.8121,0.1926,0.1083,0.9980,0.5531,0.9007,0.8667,13,2,976,107,0.5277
9,3,Random search,R4,0.4969,0.8013,0.2676,0.1583,0.9969,0.5776,0.9053,0.8636,19,3,975,101,0.4608


In [7]:
metrics = ["AP", "ROC_AUC", "F1", "Sensitivity", "Specificity", "Balanced_Accuracy", "Accuracy", "Precision", "fit_time_s"]
order = ["Manual search", "Random search", "LogReg", "Dummy"]
summ = pd.DataFrame({m: outer_res.groupby("method")[m].apply(P.mean_sd) for m in metrics}).loc[order]
num_summ = outer_res.groupby("method")[metrics].agg(["mean", lambda s: s.std(ddof=1)]).loc[order]
num_summ.columns = [f"{a}_{'mean' if b == 'mean' else 'sd'}" for a, b in num_summ.columns]
num_summ.to_csv(P.RESULTS / "outer_cv_summary.csv")
print("Dış CV özeti: Mean ± SD (n = 3 kat, örnek SD)")
summ

Dış CV özeti: Mean ± SD (n = 3 kat, örnek SD)


,AP,ROC_AUC,F1,Sensitivity,Specificity,Balanced_Accuracy,Accuracy,Precision,fit_time_s
method,,,,,,,,,
Manual search,0.4400 ± 0.0586,0.7750 ± 0.0357,0.2313 ± 0.0893,0.1466 ± 0.0736,0.9891 ± 0.0087,0.5678 ± 0.0333,0.8968 ± 0.0047,0.6614 ± 0.1816,0.6657 ± 0.1969
Random search,0.4471 ± 0.0558,0.7734 ± 0.0267,0.3061 ± 0.0467,0.1993 ± 0.0407,0.9881 ± 0.0087,0.5937 ± 0.0184,0.9017 ± 0.0063,0.7055 ± 0.1591,0.6808 ± 0.2316
LogReg,0.4598 ± 0.0743,0.7748 ± 0.0426,0.3424 ± 0.0347,0.2326 ± 0.0156,0.9840 ± 0.0098,0.6083 ± 0.0116,0.9017 ± 0.0099,0.6606 ± 0.1511,0.0322 ± 0.0035
Dummy,0.1096 ± 0.0005,0.5000 ± 0.0000,0.0000 ± 0.0000,0.0000 ± 0.0000,1.0000 ± 0.0000,0.5000 ± 0.0000,0.8904 ± 0.0005,0.0000 ± 0.0000,0.0005 ± 0.0000


## 5. Tüm development kümesinde iç CV tekrarı ve nihai seçim (3 iç kat × 12 aday = 36 eğitim)

In [8]:
dev_splits = list(N.inner_cv().split(X, y))
ctx = {"stage": "development_final", "outer_fold": 0}
runs_m = N.run_inner_search(X, y, MANUAL, dev_splits, {**ctx, "search": "manual"}, histories)
runs_r = N.run_inner_search(X, y, RANDOM, dev_splits, {**ctx, "search": "random"}, histories)
sm, sr = N.summarize_search(runs_m, MANUAL), N.summarize_search(runs_r, RANDOM)
win_m, win_r = N.select_candidate(sm), N.select_candidate(sr)
sm.insert(0, "outer_fold", "development"); sr.insert(0, "outer_fold", "development")
sm["selected"] = sm.index == win_m; sr["selected"] = sr.index == win_r
both = pd.concat([sm.loc[[win_m]], sr.loc[[win_r]]])
final_code = N.select_candidate(both)
final_method = "Manual search" if final_code.startswith("M") else "Random search"
print(f"Manuel kazanan: {win_m} (iç CV AP {sm.loc[win_m, 'mean_AP']:.4f} ± {sm.loc[win_m, 'sd_AP']:.4f})")
print(f"Otomatik kazanan: {win_r} (iç CV AP {sr.loc[win_r, 'mean_AP']:.4f} ± {sr.loc[win_r, 'sd_AP']:.4f})")
print(f"Nihai yöntem: {final_method} -> nihai konfigürasyon {final_code}")
manual_all = pd.concat([manual_outer, sm]); manual_all.index.name = "candidate"
random_all = pd.concat([random_outer, sr]); random_all.index.name = "candidate"
manual_all.to_csv(P.RESULTS / "manual_search_results.csv")
random_all.to_csv(P.RESULTS / "random_search_results.csv")
final_cfg = {kk: v for kk, v in {**MANUAL, **RANDOM}[final_code].items() if kk != "purpose"}
P.save_json({"final_candidate": final_code, "final_method": final_method,
             "manual_winner": win_m, "random_winner": win_r,
             "dev_inner_cv_mean_AP": {win_m: sm.loc[win_m, "mean_AP"], win_r: sr.loc[win_r, "mean_AP"]},
             "dev_inner_cv_sd_AP": {win_m: sm.loc[win_m, "sd_AP"], win_r: sr.loc[win_r, "sd_AP"]},
             "config": final_cfg, "threshold": 0.5, "selection_rule": SETTINGS["selection_rule"]},
            P.RESULTS / "final_selection.json")
cols_show = ["mean_AP", "sd_AP", "fold_APs", "mean_best_epoch", "mean_stopped_epoch", "n_hit_150", "total_time_s", "n_params", "selected"]
display(sm[cols_show].round(4)); display(sr[cols_show].round(4))

Manuel kazanan: M5 (iç CV AP 0.4355 ± 0.0478)
Otomatik kazanan: R1 (iç CV AP 0.4380 ± 0.0531)
Nihai yöntem: Random search -> nihai konfigürasyon R1


,mean_AP,sd_AP,fold_APs,mean_best_epoch,mean_stopped_epoch,n_hit_150,total_time_s,n_params,selected
candidate,,,,,,,,,
M1,0.4316,0.0479,0.4601;0.4585;0.3763,13.0000,28.0000,0,2.1422,2017,False
M2,0.4322,0.0456,0.4594;0.4576;0.3796,13.0000,28.0000,0,2.2173,2017,False
M3,0.4249,0.0429,0.4534;0.4458;0.3756,115.3333,125.3333,1,7.7219,2017,False
M4,0.4251,0.0534,0.4428;0.4675;0.3651,11.3333,26.3333,0,2.1463,2017,False
M5,0.4355,0.0478,0.4607;0.4654;0.3803,7.0000,22.0000,0,2.2988,6081,True
M6,0.4316,0.0479,0.4601;0.4585;0.3763,13.0000,28.0000,0,2.2533,2017,False


,mean_AP,sd_AP,fold_APs,mean_best_epoch,mean_stopped_epoch,n_hit_150,total_time_s,n_params,selected
candidate,,,,,,,,,
R1,0.4380,0.0531,0.4657;0.4716;0.3768,9.0000,24.0000,0,2.1956,6081,True
R2,0.4236,0.0495,0.4396;0.4632;0.3682,6.0000,21.0000,0,1.9032,2017,False
R3,0.4249,0.0524,0.4312;0.4738;0.3696,74.6667,89.6667,0,11.3703,1009,False
R4,0.4319,0.0471,0.4500;0.4672;0.3784,16.0000,31.0000,0,2.0182,4033,False
R5,0.4329,0.0585,0.4599;0.4731;0.3658,33.3333,48.3333,0,5.2222,16257,False
R6,0.4146,0.0495,0.4518;0.4335;0.3585,4.0000,19.0000,0,2.4427,1009,False


## 6. Ek analiz: ön işleme alternatifleri (seçime dahil değil)

Ödevin 6. bölümü RobustScaler'ın, kırpmanın ve "unknown → en sık sınıf" yaklaşımının da değerlendirilmesini istemektedir. Bu üç seçenek M1'den yalnızca tek bir faktör değiştirilerek, tüm development kümesindeki aynı iç katlarda çalıştırılır. Sonuçlar yalnızca betimsel olarak raporlanır; nihai seçim yukarıda zaten yapılmıştır ve bu sonuçlar hiçbir seçimi değiştirmez.

In [9]:
SUPP_C = {k: v for k, v in SUPP.items() if k.startswith("S_")}
runs_s = N.run_inner_search(X, y, SUPP_C, dev_splits, {"stage": "supplementary", "outer_fold": 0, "search": "supplementary"}, histories)
ss = N.summarize_search(runs_s, SUPP_C)
ref = sm.loc[["M1", "M4"]].drop(columns=["outer_fold", "selected"])
supp_tab = pd.concat([ref, ss])
supp_tab["change_vs_M1"] = ["temel (Standard, kırpma yok, unknown=kategori)", "scaler=MinMax", "scaler=Robust",
                            "%1-%99 kırpma", "unknown -> en sık sınıf"]
supp_tab.to_csv(P.RESULTS / "supplementary_preprocessing_results.csv")
supp_tab[["change_vs_M1", "mean_AP", "sd_AP", "fold_APs", "mean_best_epoch", "total_time_s"]].round(4)

,change_vs_M1,mean_AP,sd_AP,fold_APs,mean_best_epoch,total_time_s
candidate,,,,,,
M1,"temel (Standard, kırpma yok, unknown=kategori)",0.4316,0.0479,0.4601;0.4585;0.3763,13.0000,2.1422
M4,scaler=MinMax,0.4251,0.0534,0.4428;0.4675;0.3651,11.3333,2.1463
S_robust,scaler=Robust,0.4310,0.0507,0.4585;0.4619;0.3725,13.0000,2.3236
S_clip,%1-%99 kırpma,0.4432,0.0621,0.4609;0.4946;0.3742,14.0000,2.2864
S_unknown_mf,unknown -> en sık sınıf,0.4352,0.0467,0.4584;0.4658;0.3815,13.3333,2.2585


## 7. Kayıtların saklanması

In [10]:
all_inner = pd.concat([inner_outer_runs, runs_m, runs_r, runs_s], ignore_index=True)
all_inner.to_csv(P.RESULTS / "inner_cv_training_runs.csv", index=False)
hist = pd.concat(histories, ignore_index=True)
(P.RESULTS / "logs").mkdir(exist_ok=True)
hist.to_csv(P.RESULTS / "logs" / "training_histories_nested_cv.csv", index=False)
print("İç CV eğitim sayıları:", all_inner.groupby("stage").size().to_dict())
print("Dış kat yeniden eğitim sayısı:", int((outer_res.method.isin(["Manual search", "Random search"])).sum()))
print("Uyarı içeren eğitim sayısı:", int((all_inner["warnings"] != "").sum()))
print("Uyarı içeren eğitimler (aday bazında):")
print(all_inner[all_inner["warnings"] != ""].groupby(["stage", "candidate"]).size().to_string())
print("Logistic Regression uyarıları:", outer_res.loc[outer_res.method == "LogReg", "warnings"].tolist())
opt_time = all_inner[all_inner.stage.isin(["outer", "development_final"])].copy()
opt_time["optimizer"] = opt_time["candidate"].map(lambda c: {**MANUAL, **RANDOM}[c]["optimizer"])
ot = opt_time.groupby("optimizer").agg(n_trainings=("train_time_s", "size"), mean_time_s=("train_time_s", "mean"),
                                       total_time_s=("train_time_s", "sum"), mean_stopped_epoch=("stopped_epoch", "mean"),
                                       n_hit_150=("early_stopped", lambda v: int((~v).sum())))
ot.to_csv(P.RESULTS / "optimizer_training_times.csv")
ot.round(3)

İç CV eğitim sayıları: {'development_final': 36, 'outer': 108, 'supplementary': 9}
Dış kat yeniden eğitim sayısı: 6
Uyarı içeren eğitim sayısı: 10
Uyarı içeren eğitimler (aday bazında):
stage              candidate
development_final  M3           1
outer              M3           5
                   R3           4
Logistic Regression uyarıları: ['', '', '']


,n_trainings,mean_time_s,total_time_s,mean_stopped_epoch,n_hit_150
optimizer,,,,,
adam,36,1.591,57.279,62.917,4
adamw,72,0.839,60.424,35.153,0
sgd,36,1.063,38.277,56.556,6


## 8. Manuel ve otomatik arama sonuçlarının karşılaştırmalı grafiği

In [11]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4.2), sharey=True)
stages = [(1, "Dış kat 1"), (2, "Dış kat 2"), (3, "Dış kat 3"), ("development", "Tüm development")]
for ax, (fold, title) in zip(axes, stages):
    a = manual_all[manual_all.outer_fold.astype(str) == str(fold)]
    b = random_all[random_all.outer_fold.astype(str) == str(fold)]
    codes = list(a.index) + list(b.index)
    means = list(a.mean_AP) + list(b.mean_AP); sds = list(a.sd_AP) + list(b.sd_AP)
    sel = list(a.selected) + list(b.selected)
    colors = ["#2b6cb0"] * len(a) + ["#2f855a"] * len(b)
    bars = ax.bar(codes, means, yerr=sds, color=colors, capsize=3, alpha=.85)
    for bar, s in zip(bars, sel):
        if s: bar.set_edgecolor("black"); bar.set_linewidth(2)
    ax.set_title(title, fontsize=10); ax.tick_params(axis="x", labelsize=8, rotation=90)
    ax.axhline(y.mean(), color="gray", ls=":", lw=1)
axes[0].set_ylabel("İç CV ortalama AP ± SD")
fig.suptitle("Manuel (mavi) ve otomatik (yeşil) aday sonuçları — kalın kenarlı çubuk: seçilen aday; nokta çizgi: pozitif oran")
fig.tight_layout(); fig.savefig(P.FIGURES / "fig_search_comparison.png", dpi=200); plt.close(fig)

fig, ax = plt.subplots(figsize=(6.5, 4))
x = np.arange(3); w = .38
o = outer_res.set_index(["method", "outer_fold"])["AP"]
ax.bar(x - w/2, [o["Manual search", k] for k in (1, 2, 3)], w, label="Manuel arama", color="#2b6cb0")
ax.bar(x + w/2, [o["Random search", k] for k in (1, 2, 3)], w, label="Otomatik arama", color="#2f855a")
ax.plot(x, [o["LogReg", k] for k in (1, 2, 3)], "ko--", label="Logistic Regression")
ax.plot(x, [o["Dummy", k] for k in (1, 2, 3)], "s:", color="gray", label="Dummy")
ax.set_xticks(x); ax.set_xticklabels(["Dış kat 1", "Dış kat 2", "Dış kat 3"]); ax.set_ylabel("Dış değerlendirme AP")
ax.legend(fontsize=8); ax.set_title("Dış kat AP değerleri")
fig.tight_layout(); fig.savefig(P.FIGURES / "fig_outer_cv_ap.png", dpi=200); plt.close(fig)

## 9. Örnek eğitim eğrileri (tüm development iç CV, 1. iç kat): M1, M3 ve M6

In [12]:
fig, axes = plt.subplots(2, 3, figsize=(13, 6.5), sharex="col")
for j, code in enumerate(["M1", "M3", "M6"]):
    h = hist[(hist.stage == "development_final") & (hist.candidate == code) & (hist.inner_fold == 1)]
    best = h.loc[h.monitor_loss.idxmin(), "epoch"]
    axes[0, j].plot(h.epoch, h.train_loss, label="Eğitim (fit) kaybı"); axes[0, j].plot(h.epoch, h.monitor_loss, label="İzleme kaybı")
    axes[0, j].axvline(best, color="gray", ls="--", lw=1); axes[0, j].set_title(f"{code}: en iyi epoch {best}, durma {h.epoch.max()}")
    axes[1, j].plot(h.epoch, h.lr, color="#c05621"); axes[1, j].set_yscale("log"); axes[1, j].set_xlabel("Epoch")
axes[0, 0].set_ylabel("BCE kaybı"); axes[1, 0].set_ylabel("Öğrenme oranı"); axes[0, 0].legend(fontsize=8)
fig.tight_layout(); fig.savefig(P.FIGURES / "fig_training_curves_manual_examples.png", dpi=200); plt.close(fig)
print("Toplam notebook süresi: %.0f s" % (time.perf_counter() - t_start))

Toplam notebook süresi: 170 s
